# Chapter 1.6: Feature Engineering for RecSys

## Learning Objectives

By the end of this notebook, you will be able to:

1. Identify and categorize features for recommendation systems (user, item, context)
2. Apply categorical encoding techniques: one-hot, label encoding, and hashing trick
3. Construct feature crosses (manual and automatic) for capturing interactions
4. Apply numerical feature transformations: bucketization, log transform, standardization
5. Build a complete feature preprocessing pipeline
6. Understand the role of feature stores in production systems
7. Reason about feature importance and selection for recommendation tasks

## Prerequisites

- Chapter 1.5: Deep Learning Building Blocks
- Basic understanding of categorical and numerical data
- Python dictionaries and NumPy arrays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part1/chapter_1.6_feature_engineering.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/hideak1/rec_system/raw/main/notebooks/part1/chapter_1.6_feature_engineering.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import hashlib

np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

print("All imports successful!")

## 1. Feature Categories in RecSys

### User Features
- **Demographics**: age, gender, location, language
- **Behavioral**: click history, purchase frequency, session duration
- **Aggregated**: avg rating, num reviews, preferred categories

### Item Features
- **Content**: title, description, category, tags
- **Metadata**: price, brand, release date
- **Aggregated**: avg rating, num purchases, CTR

### Context Features
- **Temporal**: time of day, day of week, season
- **Spatial**: city, country, device type
- **Session**: page referrer, search query, session depth

### Cross Features
- **User x Item**: has user seen this category before? user-brand affinity
- **User x Context**: weekend shopping patterns
- **Item x Context**: seasonal item demand

In [ ]:
# Generate synthetic user and item feature data
n_samples = 10000
n_users = 500
n_items = 200

np.random.seed(42)

# User features
user_ids = np.random.randint(0, n_users, n_samples)
user_age = np.random.randint(18, 70, n_users)
user_gender = np.random.choice(['M', 'F', 'Other'], n_users, p=[0.48, 0.48, 0.04])
user_city = np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix', 'Other'], 
                              n_users, p=[0.2, 0.15, 0.1, 0.08, 0.07, 0.4])
user_avg_rating = np.clip(np.random.normal(3.5, 0.8, n_users), 1, 5)
user_num_purchases = np.random.geometric(0.05, n_users)

# Item features
item_ids = np.random.randint(0, n_items, n_samples)
item_category = np.random.choice(['Electronics', 'Books', 'Clothing', 'Home', 'Sports'],
                                  n_items, p=[0.25, 0.2, 0.25, 0.15, 0.15])
item_price = np.exp(np.random.normal(3, 1, n_items)).clip(1, 1000)  # Log-normal
item_brand = np.random.choice([f'Brand_{i}' for i in range(50)], n_items)
item_avg_ctr = np.random.beta(2, 20, n_items)  # Most items have low CTR

# Context features
hour_of_day = np.random.randint(0, 24, n_samples)
day_of_week = np.random.randint(0, 7, n_samples)
device = np.random.choice(['mobile', 'desktop', 'tablet'], n_samples, p=[0.6, 0.3, 0.1])

# Label: click (binary)
# Generate with some realistic patterns
base_ctr = item_avg_ctr[item_ids]
# Users with more purchases have higher click rates
user_factor = np.log1p(user_num_purchases[user_ids]) / 5
# Evening hours have higher engagement
hour_factor = 0.3 * np.sin((hour_of_day - 6) * np.pi / 12)
click_prob = np.clip(base_ctr + user_factor * 0.05 + hour_factor * 0.02, 0.01, 0.5)
labels = (np.random.random(n_samples) < click_prob).astype(int)

print(f"Dataset: {n_samples} samples")
print(f"Click rate: {labels.mean():.2%}")
print(f"\nSample features for first interaction:")
print(f"  User {user_ids[0]}: age={user_age[user_ids[0]]}, gender={user_gender[user_ids[0]]}, "
      f"city={user_city[user_ids[0]]}")
print(f"  Item {item_ids[0]}: category={item_category[item_ids[0]]}, "
      f"price=${item_price[item_ids[0]]:.2f}, brand={item_brand[item_ids[0]]}")
print(f"  Context: hour={hour_of_day[0]}, day={day_of_week[0]}, device={device[0]}")
print(f"  Label: {labels[0]}")

## 2. Categorical Feature Encoding

### One-Hot Encoding
Maps each category to a binary vector.

$$
\text{"Books"} \to [0, 1, 0, 0, 0]
$$

Problem: very high-dimensional for features with many categories (e.g., user_id, item_id).

### Label Encoding
Maps each category to an integer. Used as input to embedding layers.

$$
\text{"Books"} \to 1
$$

### Feature Hashing (Hashing Trick)
Maps categories to a fixed-size vector using a hash function. Handles new categories gracefully.

$$
\text{hash("Books")} \mod N \to k
$$

> **💡 Concept:** In practice, label encoding + embedding lookup is the dominant approach in deep RecSys. One-hot is wasteful for high-cardinality features, and hashing is used when the vocabulary is too large or dynamic.

In [ ]:
class CategoricalEncoder:
    """Encode categorical features with multiple strategies."""
    
    def __init__(self, method='label'):
        self.method = method
        self.vocab = {}
        self.n_buckets = None  # For hashing
    
    def fit(self, values, n_buckets=None):
        if self.method in ['label', 'onehot']:
            unique_vals = sorted(set(values))
            self.vocab = {v: i for i, v in enumerate(unique_vals)}
        elif self.method == 'hash':
            self.n_buckets = n_buckets or 100
        return self
    
    def transform(self, values):
        if self.method == 'label':
            return np.array([self.vocab.get(v, len(self.vocab)) for v in values])
        elif self.method == 'onehot':
            n_classes = len(self.vocab)
            indices = np.array([self.vocab.get(v, n_classes) for v in values])
            onehot = np.zeros((len(values), n_classes + 1))
            onehot[np.arange(len(values)), indices] = 1
            return onehot
        elif self.method == 'hash':
            return np.array([int(hashlib.md5(str(v).encode()).hexdigest(), 16) % self.n_buckets 
                            for v in values])


# Demo all three encoding methods
sample_categories = item_category[:10]

for method in ['label', 'onehot', 'hash']:
    encoder = CategoricalEncoder(method=method)
    encoder.fit(item_category, n_buckets=10)
    encoded = encoder.transform(sample_categories)
    print(f"\n{method.upper()} Encoding:")
    if method == 'onehot':
        print(f"  Shape: {encoded.shape}")
        print(f"  Sample: {sample_categories[0]} -> {encoded[0]}")
    else:
        for cat, enc in zip(sample_categories[:3], encoded[:3]):
            print(f"  {cat} -> {enc}")

In [ ]:
# Visualize hashing collision rate
bucket_sizes = [5, 10, 20, 50, 100, 200, 500, 1000]
collision_rates = []

# Use brand names as high-cardinality example
all_brands = list(set(item_brand))

for n_buckets in bucket_sizes:
    encoder = CategoricalEncoder(method='hash')
    encoder.fit(all_brands, n_buckets=n_buckets)
    hashed = encoder.transform(all_brands)
    n_unique_hashes = len(set(hashed))
    collision_rate = 1 - n_unique_hashes / len(all_brands)
    collision_rates.append(collision_rate)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(bucket_sizes, collision_rates, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Number of Hash Buckets', fontsize=12)
ax.set_ylabel('Collision Rate', fontsize=12)
ax.set_title(f'Feature Hashing: Collision Rate vs Bucket Size\n({len(all_brands)} unique brands)', fontsize=14)
ax.set_xscale('log')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print(f"Number of unique brands: {len(all_brands)}")
print(f"With {bucket_sizes[-1]} buckets: {collision_rates[-1]:.1%} collision rate")

## 3. Feature Crossing

Feature crosses capture interactions between features that individual features cannot:

### Manual Cross
$$
\text{cross}(\text{gender}, \text{category}) = \text{"M\_Electronics"}
$$

This creates a new feature that captures the interaction between gender and category.

### Automatic Cross (Embedding-based)
Instead of explicit crossing, use embedding dot products:
$$
\text{interaction}(f_i, f_j) = \langle \mathbf{e}_{f_i}, \mathbf{e}_{f_j} \rangle
$$

This is the foundation of FM (Factorization Machines), DeepFM, DCN, and other models.

> **🔑 Pro Tip:** Manual feature crosses require domain knowledge and scale quadratically with the number of features. Deep models (DCN, AutoInt) learn crosses automatically, but hand-crafted crosses often still add value because they encode known business patterns.

In [ ]:
class FeatureCrosser:
    """Manual and hashed feature crossing."""
    
    def __init__(self, n_buckets=10000):
        self.n_buckets = n_buckets
    
    def manual_cross(self, feature_a, feature_b):
        """Create string-based feature cross."""
        return [f"{a}_{b}" for a, b in zip(feature_a, feature_b)]
    
    def hashed_cross(self, feature_a, feature_b):
        """Create hashed feature cross (fixed-size output)."""
        crosses = self.manual_cross(feature_a, feature_b)
        return np.array([int(hashlib.md5(c.encode()).hexdigest(), 16) % self.n_buckets 
                        for c in crosses])


crosser = FeatureCrosser(n_buckets=1000)

# Cross gender x category
sample_gender = user_gender[user_ids[:5]]
sample_cat = item_category[item_ids[:5]]

manual = crosser.manual_cross(sample_gender, sample_cat)
hashed = crosser.hashed_cross(sample_gender, sample_cat)

print("Manual Feature Crosses:")
for g, c, m, h in zip(sample_gender, sample_cat, manual, hashed):
    print(f"  gender={g}, category={c} -> cross={m} -> hashed={h}")

# Analyze cross feature distribution
all_crosses = crosser.manual_cross(user_gender[user_ids], item_category[item_ids])
cross_counts = Counter(all_crosses)
print(f"\nUnique cross values: {len(cross_counts)}")
print(f"Top 5: {cross_counts.most_common(5)}")

## 4. Numerical Feature Transformations

### Log Transform
For heavy-tailed features (price, counts):
$$
x' = \log(1 + x)
$$

### Bucketization (Discretization)
Convert continuous features to categories:
$$
\text{age\_bucket} = \begin{cases} 0 & \text{if age} < 25 \\ 1 & \text{if } 25 \leq \text{age} < 35 \\ \vdots \end{cases}
$$

### Standardization
$$
x' = \frac{x - \mu}{\sigma}
$$

> **💡 Concept:** Bucketization is surprisingly effective in RecSys because it allows the model to learn non-linear relationships through embeddings. A user aged 22 and 23 may behave similarly, but a user aged 22 and 55 very differently. Buckets capture this naturally.

In [ ]:
# Transform price feature
transformer = NumericalTransformer()

raw_price = item_price.copy()
log_price = transformer.log_transform(raw_price)
std_price = transformer.standardize(raw_price)
bucket_price = transformer.bucketize(raw_price, [10, 25, 50, 100, 200, 500])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(raw_price, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_title('Raw Price (log-normal distribution)')
axes[0, 0].set_xlabel('Price ($)')

axes[0, 1].hist(log_price, bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Log-Transformed Price')
axes[0, 1].set_xlabel('log(1 + price)')

axes[1, 0].hist(std_price, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Standardized Price')
axes[1, 0].set_xlabel('z-score')

axes[1, 1].hist(bucket_price, bins=len(set(bucket_price)), edgecolor='black', alpha=0.7, color='purple')
axes[1, 1].set_title('Bucketized Price')
axes[1, 1].set_xlabel('Bucket ID')
all_labels = ['<$10', '$10-25', '$25-50', '$50-100', '$100-200', '$200-500', '>$500']
unique_buckets = sorted(set(bucket_price))
axes[1, 1].set_xticks(unique_buckets)
axes[1, 1].set_xticklabels([all_labels[b] for b in unique_buckets],
                            rotation=45, fontsize=9)

plt.suptitle('Numerical Feature Transformations: Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Complete Feature Preprocessing Pipeline

Let's build a full pipeline that processes all features for a CTR prediction model.

In [ ]:
class RecSysFeaturePipeline:
    """Complete feature preprocessing pipeline for RecSys."""
    
    def __init__(self):
        self.categorical_encoders = {}
        self.numerical_transformers = {}
        self.feature_config = {}
        self.feature_dims = {}  # Track dimension of each feature
    
    def add_categorical_feature(self, name, values, method='label', n_buckets=None):
        """Register a categorical feature."""
        encoder = CategoricalEncoder(method=method)
        encoder.fit(values, n_buckets=n_buckets)
        self.categorical_encoders[name] = encoder
        
        if method == 'onehot':
            self.feature_dims[name] = len(encoder.vocab) + 1
        else:
            self.feature_dims[name] = max(encoder.transform(values)) + 1
        
        self.feature_config[name] = {'type': 'categorical', 'method': method}
    
    def add_numerical_feature(self, name, values, method='standardize', boundaries=None):
        """Register a numerical feature."""
        transformer = NumericalTransformer()
        self.numerical_transformers[name] = (transformer, method, boundaries)
        
        if method == 'bucketize':
            self.feature_dims[name] = len(boundaries) + 1
            self.feature_config[name] = {'type': 'categorical', 'method': 'label'}  # Treated as categorical
        else:
            self.feature_dims[name] = 1
            self.feature_config[name] = {'type': 'numerical', 'method': method}
    
    def transform(self, feature_dict):
        """Transform a dictionary of raw features."""
        result = {}
        
        for name, values in feature_dict.items():
            if name in self.categorical_encoders:
                result[name] = self.categorical_encoders[name].transform(values)
            elif name in self.numerical_transformers:
                transformer, method, boundaries = self.numerical_transformers[name]
                if method == 'log':
                    result[name] = transformer.log_transform(values)
                elif method == 'standardize':
                    result[name] = transformer.standardize(values)
                elif method == 'bucketize':
                    result[name] = transformer.bucketize(values, boundaries)
        
        return result
    
    def summary(self):
        """Print pipeline summary."""
        print(f"{'Feature':<20} {'Type':<15} {'Method':<15} {'Dim':<10}")
        print("-" * 60)
        for name in sorted(self.feature_dims.keys()):
            config = self.feature_config[name]
            print(f"{name:<20} {config['type']:<15} {config['method']:<15} {self.feature_dims[name]:<10}")


# Build the pipeline
pipeline = RecSysFeaturePipeline()

# Categorical features
pipeline.add_categorical_feature('user_id', np.arange(n_users), method='label')
pipeline.add_categorical_feature('item_id', np.arange(n_items), method='label')
pipeline.add_categorical_feature('gender', user_gender, method='label')
pipeline.add_categorical_feature('city', user_city, method='label')
pipeline.add_categorical_feature('category', item_category, method='label')
pipeline.add_categorical_feature('brand', item_brand, method='hash', n_buckets=100)
pipeline.add_categorical_feature('device', np.unique(device), method='label')

# Numerical features
pipeline.add_numerical_feature('age', user_age.astype(float), method='bucketize',
                                boundaries=[25, 35, 45, 55, 65])
pipeline.add_numerical_feature('price', item_price, method='log')
pipeline.add_numerical_feature('avg_rating', user_avg_rating, method='standardize')
pipeline.add_numerical_feature('num_purchases', user_num_purchases.astype(float), method='log')
pipeline.add_numerical_feature('hour', hour_of_day.astype(float), method='bucketize',
                                boundaries=[6, 12, 18, 22])

pipeline.summary()

In [ ]:
# Transform features
raw_features = {
    'user_id': user_ids,
    'item_id': item_ids,
    'gender': user_gender[user_ids],
    'city': user_city[user_ids],
    'category': item_category[item_ids],
    'brand': item_brand[item_ids],
    'device': device,
    'age': user_age[user_ids].astype(float),
    'price': item_price[item_ids],
    'avg_rating': user_avg_rating[user_ids],
    'num_purchases': user_num_purchases[user_ids].astype(float),
    'hour': hour_of_day.astype(float),
}

processed = pipeline.transform(raw_features)

print("Processed features (first 3 samples):")
for name, values in sorted(processed.items()):
    print(f"  {name}: {values[:3]}")

## 6. Feature Stores: The Production Perspective

In production, features are served from a **Feature Store** - a centralized system for managing features.

Key concepts:

| Concept | Description |
|---|---|
| **Offline features** | Computed in batch (daily): user lifetime stats, item popularity |
| **Near-real-time features** | Updated every few minutes: recent click counts |
| **Real-time features** | Computed at serving time: session depth, time since last click |
| **Feature freshness** | How recently the feature was updated |
| **Feature lineage** | Which data sources and transformations produced the feature |

> **🔑 Pro Tip:** The biggest engineering challenge in RecSys is often not the model but the feature pipeline. Features must be computed consistently between training (offline) and serving (online). Any discrepancy causes **training-serving skew**, which silently degrades model performance.

In [ ]:
# Visualize feature importance using a simple correlation analysis
# (In practice, you'd use SHAP or permutation importance)

# Convert processed features to a matrix for analysis
feature_names = ['age_bucket', 'price_log', 'avg_rating', 'num_purchases_log', 'hour_bucket']
feature_matrix = np.column_stack([
    processed['age'],
    processed['price'],
    processed['avg_rating'],
    processed['num_purchases'],
    processed['hour'],
])

# Point-biserial correlation with label
correlations = []
for i, name in enumerate(feature_names):
    corr = np.corrcoef(feature_matrix[:, i], labels)[0, 1]
    correlations.append(abs(corr))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(feature_names, correlations, color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('|Correlation| with Click Label')
ax.set_title('Feature-Label Correlation (Simple Importance Proxy)')

for bar, corr in zip(bars, correlations):
    ax.text(corr + 0.001, bar.get_y() + bar.get_height()/2, 
            f'{corr:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---

## Exercises

### 🏋️ Exercise 1: Build a Feature Preprocessing Pipeline

Extend the pipeline with additional feature engineering techniques.

In [ ]:
# TODO: Extend the RecSysFeaturePipeline to support:
# 1. Multi-hot encoding for list features (e.g., item tags: ["action", "sci-fi"])
# 2. Feature crosses (e.g., gender x category)
# 3. Temporal features (e.g., is_weekend, time_of_day_bucket)
# 4. Statistical aggregation features (e.g., user's avg price of purchased items)
#
# Create synthetic data with these features and run the pipeline.
# Print the resulting feature dimensions and sample outputs.
pass

### 🏋️ Exercise 2: Train a CTR Model with Engineered Features

Use the processed features to train a simple CTR prediction model.

In [ ]:
# TODO:
# 1. Create a PyTorch model that:
#    - Uses nn.Embedding for each categorical feature
#    - Concatenates all embeddings and numerical features
#    - Passes through an MLP to predict CTR
# 2. Train on the processed features with BCE loss
# 3. Compare AUC with and without feature crosses
# 4. Compare AUC with and without numerical feature transformations
pass

### 🏋️ Exercise 3: Feature Ablation Study

Study which features matter most for prediction quality.

In [ ]:
# TODO:
# 1. Train a baseline model with ALL features
# 2. For each feature, train a model WITHOUT that feature
# 3. The drop in AUC when removing a feature indicates its importance
# 4. Also train models with ONLY user features, ONLY item features, ONLY context features
# 5. Create a visualization showing:
#    a. Feature importance (AUC drop when removed)
#    b. Feature group importance
pass

## Summary

In this notebook, we covered:

1. **Feature categories**: User, item, context, and cross features
2. **Categorical encoding**: One-hot, label encoding, and hashing trick
3. **Feature crossing**: Manual and hashed crosses for capturing interactions
4. **Numerical transformations**: Log transform, bucketization, standardization
5. **Feature pipeline**: End-to-end preprocessing for a CTR model
6. **Feature stores**: Production concerns for feature management

### Key Takeaways

- Label encoding + embedding is the dominant approach for categorical features
- Bucketization turns numerical features into categorical ones, enabling non-linear learning
- Feature crosses capture important interactions but scale quadratically
- Training-serving skew from inconsistent feature computation is a major production issue

### Next Up

In **Chapter 1.7**, we'll see how all these components come together in a modern multi-stage recommendation pipeline.